In [ ]:
# goal for this notebook: explain cross-zone differences in expected net earnings
# per hour from the simulation using zone-level features like avg trip revenue,
# avg trip duration, expected wait time, and manhattan / airport indicators

import duckdb

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [ ]:
# project paths
project_root = Path("..")
cleaned_dir = project_root / "data" / "cleaned"
raw_dir = project_root / "data" / "raw"

# connect to DuckDB
con = duckdb.connect()

In [ ]:
# load zone lookup table

zones = con.execute(f"""
SELECT *
FROM read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv')
""").fetchdf()

zones.head()

In [ ]:
# create explanatory variables for the regression

zone_features = con.execute(f"""
SELECT
    t.PULocationID,

    AVG(total_amount) AS avg_trip_revenue,

    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0
    ) AS avg_trip_minutes,

    AVG(trip_distance) AS avg_trip_distance,

    COUNT(*) AS total_pickups

FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet') t

LEFT JOIN read_csv_auto('{raw_dir.as_posix()}/taxi_zone_lookup.csv') z
    ON t.PULocationID = z.LocationID

WHERE
    total_amount > 0
    AND z.Borough != 'EWR'
    AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19

GROUP BY
    t.PULocationID
""").fetchdf()

zone_features.head()

In [ ]:
# rebuild wait-time features

zone_hour_flows = con.execute(f"""
SELECT
    zone_id,
    hour,
    SUM(pickups) AS pickups,
    SUM(dropoffs) AS dropoffs
FROM (

    SELECT
        PULocationID AS zone_id,
        EXTRACT(hour FROM tpep_pickup_datetime) AS hour,
        COUNT(*) AS pickups,
        0 AS dropoffs
    FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_pickup_datetime) BETWEEN 8 AND 19
    GROUP BY
        PULocationID,
        hour

    UNION ALL

    SELECT
        DOLocationID AS zone_id,
        EXTRACT(hour FROM tpep_dropoff_datetime) AS hour,
        0 AS pickups,
        COUNT(*) AS dropoffs
    FROM read_parquet('{cleaned_dir.as_posix()}/yellow_tripdata_2024-*.parquet')
    WHERE
        total_amount > 0
        AND EXTRACT(hour FROM tpep_dropoff_datetime) BETWEEN 8 AND 19
    GROUP BY
        DOLocationID,
        hour

)
GROUP BY
    zone_id,
    hour
""").fetchdf()

zone_hour_flows.head()

In [ ]:
# convert flows into expected wait time

zone_hour_flows["pickups"] = zone_hour_flows["pickups"].clip(lower=1)

zone_hour_flows["dropoff_pickup_ratio"] = (
    zone_hour_flows["dropoffs"] / zone_hour_flows["pickups"]
)

wait_scale = 5.0

zone_hour_flows["expected_wait_minutes"] = (
    wait_scale * zone_hour_flows["dropoff_pickup_ratio"]
)

zone_hour_flows["expected_wait_minutes"] = (
    zone_hour_flows["expected_wait_minutes"].clip(lower=1, upper=30)
)

zone_hour_flows.head()

In [ ]:
# collapse wait time to zone level

zone_wait = (
    zone_hour_flows
    .groupby("zone_id", as_index=False)["expected_wait_minutes"]
    .mean()
    .rename(columns={"zone_id": "PULocationID"})
)

zone_wait.head()

In [ ]:
# load simulation results

zone_results = pd.read_csv(project_root / "data" / "zone_results.csv")
zone_results.head()

In [ ]:
# merge everything together

analysis_df = (
    zone_results
    .merge(zone_features, on="PULocationID", how="left")
    .merge(zone_wait, on="PULocationID", how="left")
    .merge(
        zones[["LocationID", "Borough", "Zone"]],
        left_on="PULocationID",
        right_on="LocationID",
        how="left"
    )
)

analysis_df.head()

In [ ]:
analysis_df.columns

In [ ]:
analysis_df[["Borough_x","Borough_y"]].head()

In [ ]:
# create dummy variables

analysis_df["manhattan"] = (
    analysis_df["Borough_x"] == "Manhattan"
).astype(int)

airport_zone_ids = [132, 138]

analysis_df["airport_zone"] = (
    analysis_df["PULocationID"].isin(airport_zone_ids)
).astype(int)

analysis_df[["PULocationID", "Zone_x" if "Zone_x" in analysis_df.columns else "Zone", "manhattan", "airport_zone"]].head()

In [ ]:
# clean up duplicate zone/borough columns if needed

if "Zone_x" in analysis_df.columns:
    analysis_df = analysis_df.rename(columns={"Zone_x": "Zone"})
if "Borough_x" in analysis_df.columns:
    analysis_df = analysis_df.rename(columns={"Borough_x": "Borough"})

cols_to_drop = [c for c in ["Zone_y", "Borough_y", "LocationID"] if c in analysis_df.columns]
analysis_df = analysis_df.drop(columns=cols_to_drop)

analysis_df.head()

In [ ]:
# quick descriptive check

analysis_df[[
    "expected_net_earnings_per_hour",
    "avg_trip_revenue",
    "avg_trip_minutes",
    "avg_trip_distance",
    "expected_wait_minutes",
    "total_pickups"
]].describe()

In [ ]:
# run the main regression

X = analysis_df[[
    "avg_trip_revenue",
    "avg_trip_minutes",
    "expected_wait_minutes",
    "avg_trip_distance",
    "manhattan"
]]

X = sm.add_constant(X)

y = analysis_df["expected_net_earnings_per_hour"]

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
# coefficient plot

coef = model.params
conf = model.conf_int()

coef_df = pd.DataFrame({
    "coef": coef,
    "lower": conf[0],
    "upper": conf[1]
})

coef_df = coef_df.drop("const")

plt.figure(figsize=(8,5))

plt.errorbar(
    coef_df["coef"],
    coef_df.index,
    xerr=[
        coef_df["coef"] - coef_df["lower"],
        coef_df["upper"] - coef_df["coef"]
    ],
    fmt="o"
)

plt.axvline(0, linestyle="--")

plt.xlabel("Coefficient Value")
plt.title("Regression Coefficients Explaining Taxi Earnings")

plt.show()

In [ ]:
# simple scatterplot

plt.figure(figsize=(8,6))
plt.scatter(
    analysis_df["expected_wait_minutes"],
    analysis_df["expected_net_earnings_per_hour"],
    alpha=0.7
)
plt.xlabel("Expected Wait Minutes")
plt.ylabel("Expected Net Earnings per Hour")
plt.title("Wait Time vs Expected Earnings")
plt.show()

In [ ]:
# fitted vs actual - how well does regression explain the simulation

analysis_df["fitted"] = model.fittedvalues

plt.figure(figsize=(8,6))
plt.scatter(
    analysis_df["fitted"],
    analysis_df["expected_net_earnings_per_hour"],
    alpha=0.7
)

min_val = min(analysis_df["fitted"].min(), analysis_df["expected_net_earnings_per_hour"].min())
max_val = max(analysis_df["fitted"].max(), analysis_df["expected_net_earnings_per_hour"].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Fitted Earnings")
plt.ylabel("Actual Simulated Earnings")
plt.title("Regression Fit: Actual vs Fitted")
plt.show()

In [ ]:
# trip duration vs earnings

plt.figure(figsize=(7,5))

plt.scatter(
    analysis_df["avg_trip_minutes"],
    analysis_df["expected_net_earnings_per_hour"],
    alpha=0.7
)

plt.xlabel("Average Trip Duration (minutes)")
plt.ylabel("Expected Net Earnings per Hour")

plt.title("Taxi Earnings vs Trip Duration")

plt.show()

In [ ]:
# inspect zone 93 specifically

analysis_df.loc[analysis_df["PULocationID"] == 93]

In [ ]:
# compare zone 93 to top Manhattan zones

analysis_df.sort_values(
    "expected_net_earnings_per_hour",
    ascending=False
)[[
    "PULocationID",
    "Zone",
    "Borough",
    "expected_net_earnings_per_hour",
    "avg_trip_revenue",
    "avg_trip_minutes",
    "expected_wait_minutes",
    "avg_trip_distance",
    "manhattan"
]].head(15)

In [ ]:
# highlight zone 93

plt.figure(figsize=(7,5))

plt.scatter(
    analysis_df["expected_wait_minutes"],
    analysis_df["expected_net_earnings_per_hour"],
    alpha=0.6
)

zone93 = analysis_df[analysis_df["PULocationID"] == 93]

plt.scatter(
    zone93["expected_wait_minutes"],
    zone93["expected_net_earnings_per_hour"],
    color="red",
    s=100
)

plt.xlabel("Expected Wait Time")
plt.ylabel("Expected Earnings")

plt.title("Zone 93 vs Other Zones")

plt.show()